# RAG Experimentation Notebook

This notebook implements and tests the complete RAG (Retrieval-Augmented Generation) pipeline for the Yoga Assistant system.

## Objectives

1. Load best retrieval system from experiments (Hybrid Search)
2. Implement RAG flow: retrieval → context assembly → LLM generation
3. Test with sample questions
4. Experiment with multiple LLM models


## Setup and Imports


In [11]:
import pandas as pd
import numpy as np
import os
import sys
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv
import warnings
import time

warnings.filterwarnings("ignore")

# Add parent directory to path for imports
sys.path.append("..")

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

## Load Environment Variables


In [12]:
# Load environment variables from .env file
load_dotenv()

# Get API configuration
HYPERBOLIC_API_KEY = os.getenv("HYPERBOLIC_API_KEY")
LLM_MODEL = os.getenv("LLM_MODEL", "meta-llama/Meta-Llama-3.1-70B-Instruct")

print(f"API Key loaded: {'✓' if HYPERBOLIC_API_KEY else '✗'}")
print(f"Default LLM Model: {LLM_MODEL}")

API Key loaded: ✗
Default LLM Model: meta-llama/Meta-Llama-3.1-70B-Instruct


## Set Up LLM API Connection

We'll use the OpenAI-compatible API format that works with both Hyperbolic and Nebius.


In [ ]:
from openai import OpenAI

api_key = os.getenv("LLM_API_KEY")
base_url = os.getenv("LLM_BASE_URL", "https://api.hyperbolic.xyz/v1")

# Initialize OpenAI client with Hyperbolic endpoint
client = OpenAI(api_key=api_key, base_url=base_url)

print("✓ LLM API client initialized")

✓ LLM API client initialized


## Test LLM Connection


In [15]:
def test_llm_connection(model: str = LLM_MODEL) -> bool:
    """Test if LLM API is working correctly."""
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "user", "content": "Say 'Hello' if you can hear me."}
            ],
            max_tokens=50,
            temperature=0.0,
        )

        answer = response.choices[0].message.content
        print(f"✓ LLM Response: {answer}")
        print(f"✓ Model: {model}")
        print(f"✓ Tokens used: {response.usage.total_tokens}")
        return True
    except Exception as e:
        print(f"✗ Error: {e}")
        return False


# Test the connection
test_llm_connection()

✓ LLM Response: Hello.
✓ Model: meta-llama/Meta-Llama-3.1-70B-Instruct
✓ Tokens used: 48


True

## Load Yoga Data


In [16]:
# Load yoga poses dataset
yoga_data = pd.read_csv("../data/yoga_data_merged.csv")
print(f"Loaded {len(yoga_data)} yoga poses")

# Create pose dictionary for quick lookup
pose_dict = {row["id"]: row.to_dict() for _, row in yoga_data.iterrows()}
print(f"Created pose dictionary with {len(pose_dict)} poses")

Loaded 202 yoga poses
Created pose dictionary with 202 poses


## Load Best Retrieval System

We'll use the best retrieval approach from our experiments (notebook 03):

- **Weighted Product Hybrid Search** (BM25 + Vector)
- **Alpha = 0.4** (40% BM25, 60% Vector)
- **Performance**: 76% Hit Rate, 66% MRR


In [17]:
from yoga_assistant.retrieval import create_retrieval_system

# Create the retrieval system with best configuration
print("Initializing retrieval system...\n")
retrieval_system = create_retrieval_system(pose_dict)

# Test retrieval
test_query = "What poses help with balance?"
retrieved_ids = retrieval_system.search(test_query, top_k=5)

print(f"\nTest query: '{test_query}'")
print(f"Retrieved {len(retrieved_ids)} poses:")
for i, pose_id in enumerate(retrieved_ids, 1):
    pose = pose_dict[pose_id]
    print(f"{i}. {pose['pose_name']} (ID: {pose_id}, {pose['category']})")

Initializing retrieval system...

Creating retrieval system with best configuration...
- BM25: all 7 fields
- Vector: all-mpnet-base-v2
- Hybrid: Weighted Product (alpha=0.4)

Loading embedding model: all-mpnet-base-v2...
Model loaded. Embedding dimension: 768
Generating embeddings for 202 poses...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Embeddings generated. Shape: (202, 768)
✓ Retrieval system ready!

Test query: 'What poses help with balance?'
Retrieved 5 poses:
1. Cat-Cow Pose (ID: 5, standing)
2. Warrior II on One Leg (ID: 118, balancing)
3. Peacock Pose (ID: 33, balancing)
4. River Rock (ID: 199, balancing)
5. Warrior Pose (ID: 109, standing)


## Implement Context Assembly

Convert retrieved poses into a formatted context string for the LLM.


In [18]:
def assemble_context(retrieved_pose_ids: List[int]) -> str:
    """
    Format retrieved poses into a context string for the LLM.

    Args:
        retrieved_pose_ids: List of pose IDs

    Returns:
        Formatted context string
    """
    if not retrieved_pose_ids:
        return "No relevant yoga poses found."

    context_parts = []

    for i, pose_id in enumerate(retrieved_pose_ids, 1):
        pose = pose_dict[pose_id]
        context = f"""Pose {i}: {pose['pose_name']} ({pose['sanskrit_name']})
Category: {pose['category']}
Difficulty: {pose['difficulty_level']}
Benefits: {pose['benefits']}
Contraindications: {pose['contraindications']}
Instructions: {pose['instructions']}
Modifications: {pose['modifications']}
"""
        context_parts.append(context)

    return "\n---\n".join(context_parts)


# Test context assembly
test_context = assemble_context(retrieved_ids[:2])
print("Example context (first 500 chars):")
print(test_context[:500] + "...")

Example context (first 500 chars):
Pose 1: Cat-Cow Pose (Marjaryasana-Bitilasana)
Category: standing
Difficulty: beginner
Benefits: The Cat-Cow Pose stretches the spine, neck, and torso, while also improving flexibility and reducing tension. This pose can also help to warm up the body and prepare it for more dynamic movements. Additionally, it can help to calm the mind and promote relaxation.
Contraindications: This pose is generally safe for most people, but those with severe neck injuries or cervical spine problems should avoid...


## Create Prompt Template


In [19]:
def create_prompt(question: str, context: str) -> str:
    """
    Create a prompt for the LLM with question and context.

    Args:
        question: User's question
        context: Retrieved yoga pose information

    Returns:
        Formatted prompt string
    """
    prompt = f"""You are a knowledgeable yoga instructor assistant. Answer the user's question based ONLY on the provided yoga pose information. Be accurate, helpful, and concise.

If the provided information doesn't contain the answer, say so politely and suggest the user rephrase their question.

YOGA POSE INFORMATION:
{context}

USER QUESTION:
{question}

ANSWER:"""

    return prompt


# Test prompt creation
test_prompt = create_prompt(test_query, test_context[:500])
print("Example prompt (first 600 chars):")
print(test_prompt[:600] + "...")

Example prompt (first 600 chars):
You are a knowledgeable yoga instructor assistant. Answer the user's question based ONLY on the provided yoga pose information. Be accurate, helpful, and concise.

If the provided information doesn't contain the answer, say so politely and suggest the user rephrase their question.

YOGA POSE INFORMATION:
Pose 1: Cat-Cow Pose (Marjaryasana-Bitilasana)
Category: standing
Difficulty: beginner
Benefits: The Cat-Cow Pose stretches the spine, neck, and torso, while also improving flexibility and reducing tension. This pose can also help to warm up the body and prepare it for more dynamic movements. ...


## Implement Complete RAG Pipeline


In [20]:
def rag_pipeline(
    question: str,
    model: str = LLM_MODEL,
    top_k: int = 5,
    temperature: float = 0.3,
    max_tokens: int = 500,
) -> Dict[str, Any]:
    """
    Complete RAG pipeline: retrieve → assemble context → generate answer.

    Args:
        question: User's question
        model: LLM model to use
        top_k: Number of poses to retrieve
        temperature: LLM temperature (0.0 = deterministic, 1.0 = creative)
        max_tokens: Maximum tokens in response

    Returns:
        Dictionary with answer, retrieved_poses, tokens_used, response_time
    """
    start_time = time.time()

    # Step 1: Retrieve relevant poses using hybrid search
    retrieved_ids = retrieval_system.search(question, top_k=top_k)

    if not retrieved_ids:
        return {
            "answer": "I couldn't find any relevant yoga poses for your question. Could you please rephrase or ask about a specific pose, category, or benefit?",
            "retrieved_poses": [],
            "tokens_used": 0,
            "response_time_ms": int((time.time() - start_time) * 1000),
            "model": model,
        }

    # Step 2: Assemble context
    context = assemble_context(retrieved_ids)

    # Step 3: Create prompt
    prompt = create_prompt(question, context)

    # Step 4: Call LLM
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=temperature,
        )

        answer = response.choices[0].message.content
        tokens_used = response.usage.total_tokens

    except Exception as e:
        answer = f"Error generating response: {str(e)}"
        tokens_used = 0

    response_time_ms = int((time.time() - start_time) * 1000)

    return {
        "answer": answer,
        "retrieved_poses": [
            {
                "id": pose_id,
                "pose_name": pose_dict[pose_id]["pose_name"],
                "category": pose_dict[pose_id]["category"],
                "difficulty_level": pose_dict[pose_id]["difficulty_level"],
            }
            for pose_id in retrieved_ids
        ],
        "tokens_used": tokens_used,
        "response_time_ms": response_time_ms,
        "model": model,
    }


print("✓ RAG pipeline implemented")

✓ RAG pipeline implemented


## Test RAG Pipeline with Sample Questions


In [21]:
# Test with a sample question
test_question = "What poses help with balance?"

print(f"Question: {test_question}\n")
result = rag_pipeline(test_question)

print(f"Answer:\n{result['answer']}\n")
print(f"Retrieved Poses:")
for pose in result["retrieved_poses"]:
    print(
        f"  - {pose['pose_name']} (ID: {pose['id']}, {pose['difficulty_level']})"
    )
print(f"\nTokens Used: {result['tokens_used']}")
print(f"Response Time: {result['response_time_ms']}ms")
print(f"Model: {result['model']}")

Question: What poses help with balance?

Answer:
According to the provided yoga pose information, the following poses can help with balance:

1. Warrior II on One Leg (Eka Pada Virabhadrasana II) - This pose strengthens the ankles and legs, while also improving balance and focus.
2. Peacock Pose (Mayurasana) - This pose improves balance and overall core stability, helping to build focus and concentration.
3. River Rock (Nadi Shila) - This pose improves balance and focus, while also engaging the core and strengthening the ankles.

Additionally, Warrior Pose (Virabhadrasana) can also help with balance and stability, although it is not primarily a balancing pose.

Retrieved Poses:
  - Cat-Cow Pose (ID: 5, beginner)
  - Warrior II on One Leg (ID: 118, intermediate)
  - Peacock Pose (ID: 33, intermediate)
  - River Rock (ID: 199, advanced)
  - Warrior Pose (ID: 109, beginner)

Tokens Used: 1680
Response Time: 1898ms
Model: meta-llama/Meta-Llama-3.1-70B-Instruct


In [22]:
# Test with more sample questions
sample_questions = [
    "What are the benefits of Cobra Pose?",
    "Can you recommend beginner standing poses?",
    "What poses should I avoid if I have a back injury?",
    "How do I do Tree Pose?",
]

print("Testing RAG pipeline with multiple questions:\n")
print("=" * 80)

for i, question in enumerate(sample_questions, 1):
    print(f"\n{i}. Question: {question}")
    result = rag_pipeline(question)
    print(
        f"\nAnswer: {result['answer'][:300]}..."
        if len(result["answer"]) > 300
        else f"\nAnswer: {result['answer']}"
    )
    print(
        f"Retrieved: {len(result['retrieved_poses'])} poses | Tokens: {result['tokens_used']} | Time: {result['response_time_ms']}ms"
    )
    print("=" * 80)

Testing RAG pipeline with multiple questions:


1. Question: What are the benefits of Cobra Pose?

Answer: The benefits of Cobra Pose (Bhujangasana) include strengthening the back muscles, opening the chest, and improving flexibility in the shoulders and upper back. It also helps to relieve stress and anxiety, promoting a sense of calm and relaxation. Regular practice of this pose can also improve breath...
Retrieved: 5 poses | Tokens: 1776 | Time: 1104ms

2. Question: Can you recommend beginner standing poses?

Answer: Based on the provided yoga pose information, I recommend the following beginner standing poses:

1. Cat-Cow Pose (Marjaryasana-Bitilasana) - This pose is great for warming up the body and preparing it for more dynamic movements.
2. Warrior Pose I (Virabhadrasana I) - This pose strengthens the legs, ...
Retrieved: 5 poses | Tokens: 1761 | Time: 1420ms

3. Question: What poses should I avoid if I have a back injury?

Answer: Based on the provided yoga pose information, if

## Next Steps

In subsequent experiments, we will:

- Test multiple LLM models (DeepSeek-R1, DeepSeek-V3, Qwen, Llama)
- Experiment with different prompt templates
- Implement and evaluate query rewriting
- Implement and evaluate document re-ranking
- Evaluate answer quality using LLM-as-a-Judge
